# Conjuncture — Market Analysis Starter

This notebook walks through the full BidSight analysis pipeline on Thailand
government procurement data:

1. Load contract data from a local export or Firestore
2. Build benchmark tables (the same tables the live site uses)
3. Explore the discount distribution by category, agency, province
4. Simulate a bid — interactive or batch
5. Plot win curves and market concentration

**Prerequisites:**
```bash
pip install pandas pyarrow matplotlib seaborn scipy scikit-learn tqdm
```

**Get data first (run from `bidsight_scraper/`):**
```bash
python3 export_contracts.py --collection cgd_contracts --format parquet
```

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add bidsight_scraper to path so we can import bidsight_core
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'bidsight_scraper'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import bidsight_core as bc

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
print(f'bidsight_core loaded. GLOBAL_MEDIAN={bc.GLOBAL_MEDIAN}, MIN_N={bc.MIN_N}')

## 1. Load data

In [ ]:
# ── Option A: from local Parquet (recommended, fast, no reads) ──
DATA_PATH = REPO_ROOT / 'bidsight_scraper' / 'output' / 'exports' / 'cgd_contracts.parquet'

if DATA_PATH.exists():
    df_raw = pd.read_parquet(DATA_PATH)
    print(f'Loaded {len(df_raw):,} rows from {DATA_PATH.name}')
else:
    print(f'File not found: {DATA_PATH}')
    print('Run: cd bidsight_scraper && python3 export_contracts.py')
    print('Or switch to Option B below to fetch from Firestore.')
    df_raw = pd.DataFrame()

# ── Option B: from Firestore (costs reads — use sparingly) ──
# sys.path.insert(0, str(REPO_ROOT / 'bidsight_scraper'))
# from export_contracts import load_env_local, _make_firestore_client, fetch_collection
# load_env_local()
# db = _make_firestore_client()
# records = fetch_collection(db, 'cgd_contracts', limit=10000)
# df_raw = pd.DataFrame(records)

df_raw.head(3)

## 2. Clean & filter

In [ ]:
# Normalise column names (Firestore uses camelCase, exports may preserve it)
def _col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(dtype=float, name=candidates[0])

df = df_raw.copy()

df['discount']  = pd.to_numeric(_col(df, 'discountFromReference', 'discount_from_reference'), errors='coerce')
df['ref_price'] = pd.to_numeric(_col(df, 'referencePrice', 'reference_price', 'budget'), errors='coerce')
df['category']  = _col(df, 'projectType', 'project_type', 'category').astype(str).str.strip()
df['agency']    = _col(df, 'agency', 'department').astype(str).str.strip()
df['province']  = _col(df, 'province').astype(str).str.strip()
df['method']    = _col(df, 'procurementMethod', 'procurement_method').astype(str)
df['fy']        = pd.to_numeric(_col(df, 'fiscalYear', 'fiscal_year'), errors='coerce')

# e-Bidding only (competitive)
is_ebidding = df['method'].str.contains('ประกวดราคา|e-bidding|คัดเลือก', case=False, na=False)

df_clean = df[
    is_ebidding &
    df['discount'].between(0, 99.9) &
    df['ref_price'].gt(0)
].copy()

print(f'Rows after cleaning: {len(df_clean):,} (from {len(df):,} total)')
print(f'Fiscal years: {df_clean["fy"].min():.0f} – {df_clean["fy"].max():.0f}')
df_clean[['discount', 'ref_price', 'category', 'agency', 'fy']].describe()

## 3. Build benchmark tables

In [ ]:
# Convert DataFrame rows to the dict format bidsight_core expects
contracts = df_clean.rename(columns={
    'discount':  'discountFromReference',
    'ref_price': 'referencePrice',
    'category':  'projectType',
    'method':    'procurementMethod',
    'fy':        'fiscalYear',
}).to_dict('records')

print('Building benchmark tables…', end=' ', flush=True)
tables = bc.build_benchmark_tables(contracts)
print('done.')

print(f'  Category buckets:          {len(tables.category):,}')
print(f'  Agency × category buckets: {len(tables.agency_category):,}')
print(f'  Category × tier buckets:   {len(tables.category_budget_tier):,}')
print(f'  Global median discount:    {tables.global_table.median:.1f}%')

## 4. Top categories by volume

In [ ]:
cat_summary = (
    pd.DataFrame([
        {
            'category': cat,
            'n': qt.n,
            'median_disc': qt.median,
            'p10': qt.p10,
            'p90': qt.p90,
            'sigma': qt.sigma,
            'e_noc': qt.e_noc,
        }
        for cat, qt in tables.category.items()
    ])
    .sort_values('n', ascending=False)
    .reset_index(drop=True)
)

print('Top 20 categories by contract volume:')
cat_summary.head(20)

In [ ]:
# Chart: median discount by top-15 categories
top15 = cat_summary.head(15).sort_values('median_disc', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15['category'], top15['median_disc'], color='#2563EB', alpha=0.85)
ax.errorbar(
    top15['median_disc'],
    range(len(top15)),
    xerr=[top15['median_disc'] - top15['p10'], top15['p90'] - top15['median_disc']],
    fmt='none', color='#64748B', alpha=0.5, linewidth=1.2, capsize=4
)
ax.set_xlabel('Median discount from reference price (%)')
ax.set_title('Median Winning Discount by Category (p10–p90 range)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.tight_layout()
plt.show()

## 5. Discount distribution over time

In [ ]:
fy_summary = (
    df_clean
    .groupby('fy')['discount']
    .agg(['median', 'mean', 'count'])
    .reset_index()
    .rename(columns={'fy': 'Fiscal Year', 'median': 'Median %', 'mean': 'Mean %', 'count': 'Contracts'})
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(fy_summary['Fiscal Year'], fy_summary['Median %'], 'o-', color='#2563EB', label='Median discount %')
ax1.plot(fy_summary['Fiscal Year'], fy_summary['Mean %'],   's--', color='#64748B', label='Mean discount %')
ax2.bar(fy_summary['Fiscal Year'], fy_summary['Contracts'], alpha=0.15, color='#2563EB', label='Contract count')

ax1.set_xlabel('Fiscal Year (Buddhist Era)')
ax1.set_ylabel('Discount (%)')
ax2.set_ylabel('Contract count')
ax1.set_title('Thai Procurement Discount Trend — 218× Collapse in 6 Years', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(fy_summary.to_string(index=False))

## 6. Simulate a bid

In [ ]:
# ── Inputs — edit these ──────────────────────────────────────────────────────
YOUR_CATEGORY    = 'ก่อสร้าง'   # Thai category name (must match Firestore)
YOUR_AGENCY      = None           # Set to agency name for tighter benchmark, or None
YOUR_REF_PRICE   = 10.0           # Reference price (million THB)
YOUR_COST        = 8.5            # Estimated cost (million THB)
YOUR_MARGIN_FLOOR = 10.0          # Minimum acceptable gross margin (%)
YOUR_TARGET_POS  = 50.0           # Target percentile position (50 = median)
# ────────────────────────────────────────────────────────────────────────────

bench, fallback_used = bc.get_benchmark_from_tables(
    tables,
    agency=YOUR_AGENCY,
    category=YOUR_CATEGORY,
    ref_price=YOUR_REF_PRICE,
)

rec = bc.recommend_bid(
    ref_price=YOUR_REF_PRICE,
    cost_m=YOUR_COST,
    target_margin_pct=YOUR_MARGIN_FLOOR,
    benchmark=bench,
    target_position_pct=YOUR_TARGET_POS,
)

print(f'Category  : {YOUR_CATEGORY}')
print(f'Scope     : {rec.scope} (fallback={fallback_used})')
print(f'Comparable: {rec.comparable_n:,} contracts')
print()
print(f'Recommended bid      : {rec.recommended_bid:.2f} M THB  ({rec.recommended_discount:.1f}% off ref)')
print(f'Expected margin      : {rec.expected_margin:.1f}%')
print(f'Positioning          : {rec.positioning_pct}th pct  ({rec.positioning_label_en})')
print(f'Market median disc.  : {rec.market_median_discount:.1f}%')
print(f'Band (p10/p50/p90)   : {rec.band_p10:.1f} / {rec.band_median:.1f} / {rec.band_p90:.1f}%')
print()
if rec.margin_floor_breached:
    print('⚠ Margin floor breached — market is too aggressive for your cost structure.')
if rec.cannot_meet_margin:
    print('✗ Cannot meet margin at any competitive price.')
print(f'Rationale: {rec.bid_signals.rationale}')

## 7. Win curve plot

In [ ]:
curve = bc.build_curve_from_band(
    p10=rec.band_p10, p25=rec.band_p25, median=rec.band_median,
    p75=rec.band_p75, p90=rec.band_p90, n=40,
)

x = [pt['disc'] for pt in curve]
y = [pt['position_pct'] for pt in curve]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, y, '-', color='#2563EB', linewidth=2.5, label='Win curve (CDF)')
ax.axvline(rec.recommended_discount, color='#16A34A', linestyle='--', linewidth=1.8,
           label=f'Recommended bid ({rec.recommended_discount:.1f}%)')
ax.axvline(rec.band_median, color='#64748B', linestyle=':', linewidth=1.5,
           label=f'Market median ({rec.band_median:.1f}%)')

ax.axhspan(25, 75, alpha=0.06, color='#2563EB')
ax.text(rec.recommended_discount + 0.2, rec.positioning_pct + 2,
        f'{rec.positioning_pct}th pct', fontsize=9, color='#16A34A')

ax.set_xlabel('Discount from reference price (%)')
ax.set_ylabel('Position vs historical winners (%)')
ax.set_title(f'Win Curve — {YOUR_CATEGORY} ({rec.comparable_n:,} contracts)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}%'))
plt.tight_layout()
plt.show()

## 8. Market concentration (HHI)

In [ ]:
conc_rows = [
    {'category': cat, 'hhi': qt.hhi, 'e_noc': qt.e_noc, 'n': qt.n, 'median_disc': qt.median}
    for cat, qt in tables.category.items()
    if qt.hhi is not None
]
df_conc = pd.DataFrame(conc_rows).sort_values('hhi', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HHI chart
top10_hhi = df_conc.head(10)
axes[0].barh(top10_hhi['category'], top10_hhi['hhi'], color='#DC2626', alpha=0.8)
axes[0].axvline(2500, color='black', linestyle='--', linewidth=1, label='High concentration (HHI 2500)')
axes[0].set_xlabel('HHI (0 = perfect competition, 10000 = monopoly)')
axes[0].set_title('Most Concentrated Markets', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8)

# HHI vs median discount scatter
axes[1].scatter(df_conc['e_noc'], df_conc['median_disc'],
                s=df_conc['n'].clip(0, 500) * 0.5, alpha=0.5, color='#2563EB')
axes[1].set_xlabel('Effective number of competitors (eNoc)')
axes[1].set_ylabel('Median discount (%)')
axes[1].set_title('More Competitors → Higher Discounts', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 20)

plt.tight_layout()
plt.show()

print('Most concentrated categories (HHI top 5):')
print(df_conc[['category', 'hhi', 'e_noc', 'n', 'median_disc']].head(5).to_string(index=False))

## 9. Batch bid simulation

In [ ]:
# Run recommend_bid for a grid of (category, ref_price) pairs
grid = [
    {'category': 'ก่อสร้าง',      'ref_price': 5.0,   'cost_ratio': 0.88},
    {'category': 'ก่อสร้าง',      'ref_price': 50.0,  'cost_ratio': 0.88},
    {'category': 'จัดซื้อ',        'ref_price': 2.0,   'cost_ratio': 0.82},
    {'category': 'ระบบสาธารณูปโภค','ref_price': 20.0,  'cost_ratio': 0.90},
    {'category': 'ซ่อมแซม',        'ref_price': 1.0,   'cost_ratio': 0.85},
]

results = []
for g in grid:
    bench, fallback = bc.get_benchmark_from_tables(
        tables, category=g['category'], ref_price=g['ref_price']
    )
    r = bc.recommend_bid(
        ref_price=g['ref_price'],
        cost_m=g['ref_price'] * g['cost_ratio'],
        target_margin_pct=10.0,
        benchmark=bench,
    )
    results.append({
        'Category':    g['category'],
        'Ref price M': g['ref_price'],
        'Rec bid M':   r.recommended_bid,
        'Disc %':      r.recommended_discount,
        'Margin %':    r.expected_margin,
        'Position':    r.positioning_pct,
        'N':           r.comparable_n,
        'Scope':       r.scope,
    })

pd.DataFrame(results)

## 10. Export for further analysis

In [ ]:
# Export the category summary to CSV for use in other tools
out = Path('../bidsight_scraper/output/exports/category_benchmarks.csv')
out.parent.mkdir(parents=True, exist_ok=True)
cat_summary.to_csv(out, index=False, encoding='utf-8-sig')
print(f'Exported {len(cat_summary):,} rows → {out}')